# 09 — Evaluate the two-stage TCN on locked GainsByBrains

Upload the report created by Notebook 08. This CPU notebook downloads the same frozen GainsByBrains video, extracts MediaPipe landmarks, runs the supported/unknown gate and exercise classifier, scores predictions against frozen human labels, and exports a complete external report.

Use a **CPU runtime**. Do not spend T4 units on this notebook.


In [ ]:
%pip install -q yt-dlp mediapipe==0.10.35

from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import urllib.request
import zipfile

import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np
import pandas as pd
import torch
from google.colab import files
from torch import nn
from tqdm.auto import tqdm

TARGET_FPS = 15
STRIDE = 16
SMOOTH_WINDOWS = 5
MIN_SEGMENT_SECONDS = 2.0
DEVICE = torch.device("cpu")
print("Device:", DEVICE)
print("MediaPipe:", mp.__version__)


In [ ]:
# The first run uses the freshly downloaded Notebook 08 report.
# After the candidate is archived, this can be switched to a GitHub URL.
DOWNLOAD_MODEL_FROM_GITHUB = False
MODEL_ZIP_URL = ""

if DOWNLOAD_MODEL_FROM_GITHUB:
    model_zip = Path("/content/calorie_two_stage_tcn_report.zip")
    request = urllib.request.Request(MODEL_ZIP_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=120) as response, model_zip.open("wb") as output:
        shutil.copyfileobj(response, output)
else:
    print("Upload calorie_two_stage_tcn_report.zip")
    uploaded = files.upload()
    model_zip = next((Path(name) for name in uploaded if name.lower().endswith(".zip")), None)
    assert model_zip, "Upload the report ZIP made by Notebook 08."

MODEL_ROOT = Path("/content/two_stage_tcn_report")
if MODEL_ROOT.exists():
    shutil.rmtree(MODEL_ROOT)
MODEL_ROOT.mkdir()
with zipfile.ZipFile(model_zip) as archive:
    archive.extractall(MODEL_ROOT)
MODEL_FILE = next(MODEL_ROOT.rglob("two_stage_tcn.pt"), None)
assert MODEL_FILE, "two_stage_tcn.pt was not found in the report ZIP."
print("Model:", MODEL_FILE)


In [ ]:
VIDEO_ID = "MlQCrNNkl48"
VIDEO_PATH = Path("/content/gainsbybrains__MlQCrNNkl48.mp4")
subprocess.run([
    "yt-dlp", "--no-playlist",
    "-f", "134/best[height<=360][ext=mp4]/best[height<=360]",
    "-o", str(VIDEO_PATH),
    f"https://www.youtube.com/watch?v={VIDEO_ID}",
], check=True)
print("Video:", VIDEO_PATH, round(VIDEO_PATH.stat().st_size / 1024**2, 2), "MB")

POSE_MODEL_URL = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task"
POSE_MODEL_PATH = Path("/content/pose_landmarker_full.task")
if not POSE_MODEL_PATH.exists():
    urllib.request.urlretrieve(POSE_MODEL_URL, POSE_MODEL_PATH)
print("Pose model ready:", POSE_MODEL_PATH.exists())


In [ ]:
class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation, dropout=0.2):
        super().__init__()
        padding = dilation
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
        )
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return torch.relu(self.net(x) + self.residual(x))


class SmallTCN(nn.Module):
    def __init__(self, input_features, num_classes):
        super().__init__()
        channels = [64, 96, 128, 128]
        blocks, current = [], input_features
        for level, output in enumerate(channels):
            blocks.append(TemporalBlock(current, output, dilation=2 ** level))
            current = output
        self.features = nn.Sequential(*blocks)
        self.classifier = nn.Linear(current, num_classes)

    def forward(self, x):
        return self.classifier(self.features(x).mean(dim=-1))


checkpoint = torch.load(MODEL_FILE, map_location=DEVICE, weights_only=False)
ALL_CLASSES = list(checkpoint["classes"])
SUPPORTED_CLASSES = list(checkpoint["supported_classes"])
GATE_THRESHOLD = float(checkpoint["gate_threshold"])
WINDOW = int(checkpoint.get("window_frames", 64))
input_features = int(checkpoint.get("input_features", 34))

gate_model = SmallTCN(input_features, 2).to(DEVICE)
exercise_model = SmallTCN(input_features, len(SUPPORTED_CLASSES)).to(DEVICE)
gate_model.load_state_dict(checkpoint["gate_state_dict"])
exercise_model.load_state_dict(checkpoint["exercise_state_dict"])
gate_model.eval()
exercise_model.eval()
print("Gate threshold:", GATE_THRESHOLD)
print("Supported classes:", SUPPORTED_CLASSES)


In [ ]:
KEEP_17 = [0, 2, 5, 7, 8, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
LEFT_SHOULDER, RIGHT_SHOULDER = 5, 6
LEFT_HIP, RIGHT_HIP = 11, 12


def normalize_pose(frame):
    xyz = frame[:, :3].copy()
    hip_center = (xyz[LEFT_HIP] + xyz[RIGHT_HIP]) / 2.0
    shoulder_center = (xyz[LEFT_SHOULDER] + xyz[RIGHT_SHOULDER]) / 2.0
    scale = max(
        float(np.linalg.norm(xyz[LEFT_SHOULDER] - xyz[RIGHT_SHOULDER])),
        float(np.linalg.norm(shoulder_center - hip_center)),
        1e-6,
    )
    xyz = (xyz - hip_center) / scale
    return np.concatenate([xyz, frame[:, 3:4]], axis=1)


def fill_missing(sequence):
    flat = sequence.reshape(len(sequence), -1)
    filled = pd.DataFrame(flat).interpolate(limit_direction="both").fillna(0).to_numpy()
    return filled.reshape(sequence.shape).astype(np.float32)


cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened(), "OpenCV could not open the video."
source_fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
source_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
sample_interval = 1.0 / TARGET_FPS
next_sample_s = 0.0
frame_index = 0
poses, times, detected = [], [], 0
options = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(POSE_MODEL_PATH)),
    running_mode=mp.tasks.vision.RunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_segmentation_masks=False,
)
progress = tqdm(total=source_frames if source_frames else None, desc="Reading video frames")
with mp.tasks.vision.PoseLandmarker.create_from_options(options) as landmarker:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        time_s = frame_index / source_fps
        frame_index += 1
        progress.update(1)
        if time_s + 1e-9 < next_sample_s:
            continue
        next_sample_s += sample_interval
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = landmarker.detect_for_video(
            mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb),
            int(round(time_s * 1000)),
        )
        if result.pose_landmarks:
            landmarks = result.pose_landmarks[0]
            pose = np.asarray([
                [landmarks[i].x, landmarks[i].y, landmarks[i].z, float(landmarks[i].visibility or 0.0)]
                for i in KEEP_17
            ], dtype=np.float32)
            poses.append(normalize_pose(pose))
            detected += 1
        else:
            poses.append(np.full((17, 4), np.nan, dtype=np.float32))
        times.append(time_s)
progress.close()
cap.release()
assert poses, "No frames were sampled."
sequence = fill_missing(np.stack(poses))
detection_rate = detected / len(sequence)
print("Landmark frames:", len(sequence), "detection:", round(detection_rate, 3))
assert detection_rate >= 0.5


In [ ]:
starts = list(range(0, max(1, len(sequence) - WINDOW + 1), STRIDE))
if len(sequence) > WINDOW and starts[-1] != len(sequence) - WINDOW:
    starts.append(len(sequence) - WINDOW)
features, centers = [], []
for start in starts:
    window = sequence[start:start + WINDOW]
    if len(window) < WINDOW:
        window = np.concatenate([window, np.repeat(window[-1:], WINDOW - len(window), axis=0)])
    features.append(window[:, :, :2].reshape(WINDOW, -1).T)
    centers.append(min((start + WINDOW / 2) / TARGET_FPS, times[-1]))

batch = torch.from_numpy(np.asarray(features, dtype=np.float32))
gate_parts, exercise_parts = [], []
with torch.no_grad():
    for start in range(0, len(batch), 256):
        current = batch[start:start + 256].to(DEVICE)
        gate_parts.append(gate_model(current).softmax(1).cpu().numpy())
        exercise_parts.append(exercise_model(current).softmax(1).cpu().numpy())
gate_probability = np.concatenate(gate_parts)
exercise_probability = np.concatenate(exercise_parts)

# Smooth both stages over neighboring windows.
gate_smoothed = pd.DataFrame(gate_probability).rolling(
    SMOOTH_WINDOWS, center=True, min_periods=1
).mean().to_numpy()
exercise_smoothed = pd.DataFrame(exercise_probability).rolling(
    SMOOTH_WINDOWS, center=True, min_periods=1
).mean().to_numpy()
exercise_ids = exercise_smoothed.argmax(axis=1)
predicted_labels = np.asarray([
    SUPPORTED_CLASSES[class_id] if supported_probability >= GATE_THRESHOLD else "unknown"
    for supported_probability, class_id in zip(gate_smoothed[:, 1], exercise_ids)
])
confidences = np.asarray([
    supported_probability * exercise_smoothed[index, class_id]
    if predicted_labels[index] != "unknown" else 1.0 - supported_probability
    for index, (supported_probability, class_id) in enumerate(zip(gate_smoothed[:, 1], exercise_ids))
])
window_predictions = pd.DataFrame({
    "center_s": centers,
    "predicted_class": predicted_labels,
    "confidence": confidences,
    "gate_supported_probability": gate_smoothed[:, 1],
    "exercise_prediction": [SUPPORTED_CLASSES[index] for index in exercise_ids],
    "exercise_confidence": exercise_smoothed.max(axis=1),
})
display(window_predictions.head(10))


In [ ]:
def build_segments(predictions, video_end_s):
    centers_array = predictions["center_s"].to_numpy(dtype=float)
    boundaries = np.empty(len(centers_array) + 1, dtype=float)
    boundaries[0], boundaries[-1] = 0.0, video_end_s
    if len(centers_array) > 1:
        boundaries[1:-1] = (centers_array[:-1] + centers_array[1:]) / 2.0
    raw = []
    for index, row in enumerate(predictions.itertuples(index=False)):
        if raw and raw[-1]["label"] == row.predicted_class:
            raw[-1]["end_s"] = boundaries[index + 1]
            raw[-1]["confidences"].append(row.confidence)
        else:
            raw.append({
                "label": row.predicted_class,
                "start_s": boundaries[index],
                "end_s": boundaries[index + 1],
                "confidences": [row.confidence],
            })
    for segment in raw:
        if segment["end_s"] - segment["start_s"] < MIN_SEGMENT_SECONDS:
            segment["label"] = "unknown"
    merged = []
    for segment in raw:
        if merged and merged[-1]["label"] == segment["label"]:
            merged[-1]["end_s"] = segment["end_s"]
            merged[-1]["confidences"].extend(segment["confidences"])
        else:
            merged.append(segment)
    return pd.DataFrame([{
        "start_s": round(item["start_s"], 2),
        "end_s": round(item["end_s"], 2),
        "duration_s": round(item["end_s"] - item["start_s"], 2),
        "predicted_class": item["label"],
        "mean_confidence": round(float(np.mean(item["confidences"])), 3),
    } for item in merged])


timeline = build_segments(window_predictions, times[-1])
display(timeline)


In [ ]:
OUTPUT_ROOT = Path("/content/two_stage_mixed_video_analysis")
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir()
window_predictions.to_csv(OUTPUT_ROOT / "window_predictions.csv", index=False)
timeline.to_csv(OUTPUT_ROOT / "timeline.csv", index=False)

# Combined probability plot: gate probability plus the exercise probabilities
# weighted by the gate's supported probability.
plt.figure(figsize=(16, 5))
plt.plot(centers, 1.0 - gate_smoothed[:, 1], label="unknown", linewidth=1.5)
for index, class_name in enumerate(SUPPORTED_CLASSES):
    plt.plot(
        centers,
        gate_smoothed[:, 1] * exercise_smoothed[:, index],
        label=class_name,
        linewidth=1,
    )
plt.axhline(GATE_THRESHOLD, color="black", linestyle="--", linewidth=1, label="gate threshold")
plt.xlabel("Video time (seconds)")
plt.ylabel("Combined probability")
plt.title("Two-stage TCN predictions")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "probability_timeline.png", dpi=170)
plt.show()

summary = {
    "video": VIDEO_PATH.name,
    "video_source": f"https://www.youtube.com/watch?v={VIDEO_ID}",
    "video_sha256": hashlib.sha256(VIDEO_PATH.read_bytes()).hexdigest(),
    "duration_s": float(times[-1]),
    "pose_detection_rate": float(detection_rate),
    "classes": ALL_CLASSES,
    "supported_classes": SUPPORTED_CLASSES,
    "gate_threshold": GATE_THRESHOLD,
    "segments": len(timeline),
}
(OUTPUT_ROOT / "summary.json").write_text(json.dumps(summary, indent=2))

# Frozen human-label scoring.
raw_root = "https://raw.githubusercontent.com/bariscelikk1/cALorie/feat/tcn-rebuild"
labels_path = OUTPUT_ROOT / "labels_v1.csv"
scorer_path = Path("/content/score_exported_mixed_video.py")
urllib.request.urlretrieve(
    f"{raw_root}/ml/artifacts/v1/evaluation/gainsbybrains_external_v1/labels_v1.csv",
    labels_path,
)
urllib.request.urlretrieve(
    f"{raw_root}/ml/scripts/score_exported_mixed_video.py",
    scorer_path,
)
subprocess.run([
    "python", str(scorer_path),
    "--predictions", str(OUTPUT_ROOT / "window_predictions.csv"),
    "--annotations", str(labels_path),
    "--output", str(OUTPUT_ROOT),
], check=True)
external_metrics = json.loads((OUTPUT_ROOT / "external_test_metrics.json").read_text())
print(json.dumps(external_metrics, indent=2))

zip_path = shutil.make_archive(
    "/content/two_stage_mixed_video_analysis", "zip", root_dir=OUTPUT_ROOT
)
print("Analysis ZIP:", zip_path)
files.download(zip_path)
